In [1]:
# 01_limpieza_datos.ipynb
# Limpieza y unificación de CSV de criptomonedas
# Totalmente robusto ante inconsistencias de Excel

import pandas as pd
import os
from glob import glob

# -----------------------------
# 1. Configuración de carpetas
# -----------------------------
carpeta_crudos = "../datos/crudos/"
carpeta_procesados = "../datos/procesados/"

os.makedirs(carpeta_procesados, exist_ok=True)

archivos = glob(os.path.join(carpeta_crudos, "*.csv"))

if not archivos:
    raise FileNotFoundError(f"No se encontraron archivos CSV en {carpeta_crudos}")

lista_dfs = []

# -----------------------------
# 2. Lectura y limpieza robusta
# -----------------------------
for archivo in archivos:
    print(f"Procesando archivo: {archivo}")
    
    # Nombre de la cripto
    cripto_id = os.path.basename(archivo).split(".")[0].upper()
    
    # Leer CSV detectando separador automáticamente y quitando comillas
    df = pd.read_csv(archivo, encoding="utf-8", sep=None, engine='python', quotechar='"', skipinitialspace=True)
    
    # Limpiar nombres de columnas y caracteres invisibles
    df.columns = df.columns.str.strip().str.lower().str.replace('\ufeff','')  # BOM
    df = df.applymap(lambda x: str(x).strip() if isinstance(x, str) else x)
    
    # Renombrar columnas a español
    df = df.rename(columns={
        "ticker": "cripto_id",
        "date": "fecha",
        "open": "apertura",
        "high": "maximo",
        "low": "minimo",
        "close": "cierre"
    })
    
    # Sobrescribir cripto_id por seguridad
    df["cripto_id"] = cripto_id
    
    # Verificar columnas mínimas
    columnas_esperadas = ["cripto_id","fecha","apertura","maximo","minimo","cierre"]
    df = df.dropna(subset=columnas_esperadas)  # elimina filas incompletas
    
    # Convertir fecha a datetime
    df["fecha"] = pd.to_datetime(df["fecha"], dayfirst=True, errors='coerce')
    df = df.dropna(subset=["fecha"])  # eliminar fechas inválidas
    
    # Limpiar y convertir precios a float
    for col in ["apertura","maximo","minimo","cierre"]:
        df[col] = df[col].astype(str)
        df[col] = df[col].str.replace(".", "", regex=False)  # eliminar separador de miles
        df[col] = df[col].str.replace(",", ".", regex=False) # reemplazar coma decimal
        df[col] = pd.to_numeric(df[col], errors='coerce')     # convertir a float
    df = df.dropna(subset=["apertura","maximo","minimo","cierre"])  # eliminar filas inválidas
    
    # Ordenar por fecha y eliminar duplicados
    df = df.sort_values("fecha")
    df = df.drop_duplicates(subset=["cripto_id", "fecha"])
    
    # Añadir a la lista
    lista_dfs.append(df)

# -----------------------------
# 3. Concatenar todos los DataFrames
# -----------------------------
df_final = pd.concat(lista_dfs, ignore_index=True)

# -----------------------------
# 4. Guardar dataset procesado
# -----------------------------
ruta_procesado = os.path.join(carpeta_procesados, "precios_diarios.csv")
df_final.to_csv(ruta_procesado, index=False, encoding="utf-8", sep=';')

# -----------------------------
# 5. Resumen final
# -----------------------------
print("Dataset unificado y guardado en:", ruta_procesado)
print("Número de registros:", len(df_final))
print("Criptomonedas incluidas:", df_final["cripto_id"].unique())
print("Primeros registros:\n", df_final.head())


Procesando archivo: ../datos/crudos\BTC.csv
Procesando archivo: ../datos/crudos\DOGE.csv
Procesando archivo: ../datos/crudos\NEO.csv
Procesando archivo: ../datos/crudos\UNI.csv
Procesando archivo: ../datos/crudos\ZEN.csv


C:\Users\alefe\AppData\Local\Temp\ipykernel_18380\4248199213.py:38: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df = df.applymap(lambda x: str(x).strip() if isinstance(x, str) else x)
C:\Users\alefe\AppData\Local\Temp\ipykernel_18380\4248199213.py:38: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df = df.applymap(lambda x: str(x).strip() if isinstance(x, str) else x)
C:\Users\alefe\AppData\Local\Temp\ipykernel_18380\4248199213.py:38: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df = df.applymap(lambda x: str(x).strip() if isinstance(x, str) else x)
C:\Users\alefe\AppData\Local\Temp\ipykernel_18380\4248199213.py:38: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df = df.applymap(lambda x: str(x).strip() if isinstance(x, str) else x)
C:\Users\alefe\AppData\Local\Temp\ipykernel_18380\4248199213.py:38: FutureWarning: DataFrame.applymap ha

Dataset unificado y guardado en: ../datos/procesados/precios_diarios.csv
Número de registros: 16942
Criptomonedas incluidas: ['BTC' 'DOGE' 'NEO' 'UNI' 'ZEN']
Primeros registros:
   cripto_id      fecha  apertura  maximo  minimo  cierre
0       BTC 2010-07-17      4951    4951    4951    4951
1       BTC 2010-07-18      4951    8585    4951    8584
2       BTC 2010-07-19      8584    9307    7723     808
3       BTC 2010-07-20       808    8181    7426    7474
4       BTC 2010-07-21      7474    7921    6634    7921
